<a href="https://colab.research.google.com/github/SPapalonyan/task_13_3/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Анализ тональности отзывов кафе
Модель: `blanchefort/rubert-base-cased-sentiment`

In [ ]:
# Шаг 1: Установка библиотек
!pip install transformers pandas torch

In [ ]:
# Шаг 2: Загрузка датасета
import pandas as pd

URL = 'https://raw.githubusercontent.com/SPapalonyan/task_13_3/refs/heads/main/cafe_reviews.csv'
df = pd.read_csv(URL)
print(f'Загружено строк: {len(df)}')
df.head()

In [ ]:
# Шаг 3: Загрузка модели и функция analyze_sentiment
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# Маппинг: [0] = Neutral, [1] = Positive, [2] = Negative
LABEL_MAP = {0: 'Neutral', 1: 'Positive', 2: 'Negative'}

def analyze_sentiment(text):
    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return LABEL_MAP[predicted_class]

print('Модель загружена!')

In [ ]:
# Шаг 4: Применение функции ко всем отзывам
df['predicted_sentiment'] = df['text'].apply(analyze_sentiment)

print('Распределение тональностей:')
print(df['predicted_sentiment'].value_counts())
df[['text', 'predicted_sentiment']].head(10)

In [ ]:
# Шаг 5: Столбчатая диаграмма
import matplotlib.pyplot as plt

sentiment_counts = df['predicted_sentiment'].value_counts()

colors = {'Positive': '#4CAF50', 'Negative': '#F44336', 'Neutral': '#2196F3'}
bar_colors = [colors.get(label, '#9E9E9E') for label in sentiment_counts.index]

plt.figure(figsize=(8, 5))
plt.bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors, edgecolor='black')
plt.xlabel('Тональность')
plt.ylabel('Количество отзывов')
plt.title('Распределение тональности отзывов (Predicted)')
for i, (label, count) in enumerate(sentiment_counts.items()):
    plt.text(i, count + 0.5, str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()